# IFEval smoke run: judge-free grading through the ScreamingFace SDK

IFEval (arXiv:2311.07911) carries 541 prompts with machine-checkable constraints — word
counts, forbidden punctuation, required sections. The Engine grades every response with a
deterministic verifier: **no judge model, zero grading cost**.

> **Cost note:** the evaluation cell performs exactly one Candidate call per selected
> case and nothing else. Discovery makes no model calls.

## Before running

The local AI Gateway must be running on `127.0.0.1:9105`, and the isolated Engine demo must be
running on `127.0.0.1:9108`. The connection panel sends the OpenRouter key through the Engine to
AI Gateway; the Client never calls AI Gateway directly.

For a host-local Engine, prepare IFEval's pinned cases (this also downloads the offline
NLTK tokenizer corpus the verifier reads) and pass the assets root explicitly:

```bash
uv run --with datasets python -m url4_cloud.benchmarks.ifeval.prepare \
  --out /tmp/screamingface-benchmark-assets/ifeval
URL4_BENCHMARK_ASSETS=/tmp/screamingface-benchmark-assets \
  uv run url4-cloud serve --local
```

`/opt/benchmarks` is the container image default and normally does not exist on the host.

In [ ]:
import screamingface as sf

## Connect OpenRouter

In [ ]:
sf.connect()

## Meet the benchmark

Before spending anything, read what the exam actually is. Discovery is free — plain
Engine REST, no model calls.

In [ ]:
sf.benchmarks.list()

In [ ]:
ifeval = sf.benchmarks.get("ifeval")
ifeval

### Read real prompts

Each prompt carries its constraints **in its own text** — "no commas", "at least 300
words", "highlight 3 sections". That is what makes IFEval machine-checkable: the Engine's
deterministic verifier re-reads the response against exactly those constraints, so
grading needs no judge model. Page further with `ifeval.cases(limit=3, offset=100)`.

In [ ]:
ifeval.cases(limit=3)

## Define a Candidate

In [ ]:
haiku = sf.Model("openrouter/anthropic/claude-haiku-4.5")

## Evaluate the benchmark

`limit=5` selects the first five of IFEval's 541 prompts — five Candidate calls total.
The primary score is the paper's prompt-level strict accuracy; instruction-level and
loose readings arrive in the metrics.

In [ ]:
report = sf.evaluate(
    haiku,
    benchmark="ifeval",
    limit=5,
)
report

## Inspect the Report

In [ ]:
report.candidates

In [ ]:
report.usage

In [ ]:
report.to_json()

## R1 preview — probing the corrective-loop syntax (no services, no cost)

Everything above was **R0**: each candidate answers each prompt once. The LANL paper's
result (97.34% strict with small models) comes from a **corrective loop**: answer →
deterministic check → turn violations into feedback → retry, bounded at 3 attempts.

In url4 that loop *unrolls* into a nested expression with **named siblings**:

```
( prior_2:( prior_1:()/member!'attempt-1',
            grade:($prior_1)/check!'…'      ← checker reads the FIRST answer
          )/member!'attempt-2',              ← second attempt sees answer + feedback
  grade:($prior_2)/check!'…' )!'$…'
```

Two things must be true for this to work, and we can test both **right here** — the
`url4` package in this environment is the *same* DAG engine the Cloud embeds. The probe
below builds a miniature world with two fake routes (no model, no network, $0):

- `/member` — a stand-in candidate. It answers **with a comma** on attempt 1; if its
  input contains checker feedback, it corrects itself.
- `/grade` — a stand-in verifier: passes iff the answer has no comma (the real thing is
  the vendored IFEval checker behind `/check`).

What we're proving: **(a)** named siblings (`prior_1:`, `grade_1:`) execute and are
referenceable as `$prior_1`; **(b)** a `grade:` node can target a *deterministic route*,
not just a model.

In [11]:
import json

from url4 import RelExpr, Text, expr, render, src
from url4.peer.server import Request, Url4Node

probe = Url4Node("r1-probe")
# Every route call is recorded here in execution order, so each attempt's answer
# and each checker verdict stay visible after the run.
trace: list[dict] = []


@probe.endpoint("/member")
def member(request: Request) -> str:
    context = request.context or ""
    # Attempt 2 receives the attempt-1 answer + checker feedback as its input.
    if "feedback" in context.lower() and "PASSED" not in context:
        answer = "Tea is warm and nice without a single comma"
    else:
        answer = "Tea is warm, and it is nice."
    trace.append({"route": "/member", "intent": request.intent, "in": context, "out": answer})
    return answer


@probe.endpoint("/grade")
def grade(request: Request) -> str:
    answer = request.context or ""
    passed = "," not in answer
    verdict = json.dumps(
        {"passed": passed, "feedback": "PASSED" if passed else "violation: remove every comma"}
    )
    trace.append({"route": "/grade", "intent": request.intent, "in": answer, "out": verdict})
    return verdict

Build the two-attempt chain. Reading inside-out: `prior_1` answers, `grade_1`
checks it (note its input is the *reference* `$prior_1`), the whole group becomes
attempt 2's input, and `grade_2` checks the retry. The rendered string is the same
syntax shape as the full 3-attempt R1 chain.

In [12]:
def check(reference: str) -> RelExpr:
    return RelExpr(path="/grade", context=reference, intent=Text("no_comma"))


attempt_1 = expr(
    src(
        RelExpr(path="/member", context="Describe tea. No commas.", intent=Text("attempt-1")),
        name="prior_1",
        weight=0.0,
    ),
    src(check("$prior_1"), name="grade_1", weight=0.0),
    intent=Text("previous answer: $prior_1 | checker feedback: $grade_1"),
)
chain = expr(
    src(
        RelExpr(path="/member", context=render(attempt_1), intent=Text("attempt-2")),
        name="prior_2",
        weight=0.0,
    ),
    src(check("$prior_2"), name="grade_2", weight=0.0),
    intent=Text("final answer: $prior_2 | final grade: $grade_2"),
)
print(render(chain))

(prior_2:0.0:/member((prior_1:0.0:/member(Describe tea. No commas.)!'attempt-1', grade_1:0.0:/grade($prior_1)!'no_comma')!'previous answer: $prior_1 | checker feedback: $grade_1')!'attempt-2', grade_2:0.0:/grade($prior_2)!'no_comma')!'final answer: $prior_2 | final grade: $grade_2'


In [13]:
trace.clear()
result = await probe.evaluate(render(chain))
result.text

'final answer: Tea is warm and nice without a single comma | final grade: {"passed": true, "feedback": "PASSED"}'

### Watch the loop happen

The trace shows every route call in execution order — `prior_1`'s answer, its verdict,
the feedback flowing into attempt 2, and the retry's verdict:

In [14]:
for index, step in enumerate(trace, 1):
    print(f"step {index} — {step['route']} !{step['intent']}")
    print(f"   in : {step['in'][:110]}")
    print(f"   out: {step['out']}")
    print()

step 1 — /member !attempt-1
   in : Describe tea. No commas.
   out: Tea is warm, and it is nice.

step 2 — /grade !no_comma
   in : Tea is warm, and it is nice.
   out: {"passed": false, "feedback": "violation: remove every comma"}

step 3 — /member !attempt-2
   in : previous answer: Tea is warm, and it is nice. | checker feedback: {"passed": false, "feedback": "violation: re
   out: Tea is warm and nice without a single comma

step 4 — /grade !no_comma
   in : Tea is warm and nice without a single comma
   out: {"passed": true, "feedback": "PASSED"}



**How to read the trace:** step 1 — attempt 1 answers *with* a comma. Step 2 — the
checker fails it and names the exact violation. Step 3 — attempt 2's input carries the
prior answer **and** that feedback, so it corrects itself. Step 4 — the retry passes.
The corrective loop's data flow works end-to-end in today's url4 syntax.

Still open before real R1 (`OME-721`): linking the *same real candidate* into all
attempt slots, per-member checks for ensembles, and the cost caveat — an unrolled chain
runs every attempt even when attempt 1 already passed, so R1 claims accuracy only.